# NLP Feature Engineering vs Embeddings — Google Colab Notebook

This notebook compares:
1. **TF-IDF + Logistic Regression**
2. **Word Embedding averaging + Logistic Regression**
3. **Semantic similarity with embeddings**

It is designed for classroom use and small experiments.

## 1. Install dependencies

In [1]:
!pip -q install sentence-transformers scikit-learn pandas matplotlib

## 2. Small  dataset

In [2]:
import pandas as pd

data = [
    ("This product is amazing and works perfectly", "positive"),
    ("I absolutely love this phone", "positive"),
    ("Excellent quality and very fast delivery", "positive"),
    ("The movie was fantastic and emotional", "positive"),
    ("Customer support was helpful and kind", "positive"),
    ("This is the worst service I have ever used", "negative"),
    ("I hate this update", "negative"),
    ("The food was cold and tasteless", "negative"),
    ("Terrible experience, not recommended", "negative"),
    ("The laptop stopped working after one day", "negative"),
    ("The package arrived yesterday", "neutral"),
    ("The meeting is scheduled for Monday", "neutral"),
    ("This phone has a 6 inch screen", "neutral"),
    ("The train leaves at 8 PM", "neutral"),
    ("The report contains five sections", "neutral"),
]

df = pd.DataFrame(data, columns=["text", "label"])
df

,text,label
0,This product is amazing and works perfectly,positive
1,I absolutely love this phone,positive
2,Excellent quality and very fast delivery,positive
3,The movie was fantastic and emotional,positive
4,Customer support was helpful and kind,positive
5,This is the worst service I have ever used,negative
6,I hate this update,negative
7,The food was cold and tasteless,negative
8,"Terrible experience, not recommended",negative
9,The laptop stopped working after one day,negative


## 3. Train / test split

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 10
Test size: 5


## 4. Approach A — TF-IDF + Logistic Regression

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

tfidf_clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000))
])

tfidf_clf.fit(X_train, y_train)
preds_tfidf = tfidf_clf.predict(X_test)

print("TF-IDF Results")
print(classification_report(y_test, preds_tfidf))

TF-IDF Results
              precision    recall  f1-score   support

    negative       0.20      1.00      0.33         1
     neutral       0.00      0.00      0.00         2
    positive       0.00      0.00      0.00         2

    accuracy                           0.20         5
   macro avg       0.07      0.33      0.11         5
weighted avg       0.04      0.20      0.07         5



/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

## 5. Approach B — Sentence Embeddings + Logistic Regression

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedder.encode(list(X_train), convert_to_numpy=True)
X_test_emb = embedder.encode(list(X_test), convert_to_numpy=True)

emb_clf = LogisticRegression(max_iter=2000)
emb_clf.fit(X_train_emb, y_train)
preds_emb = emb_clf.predict(X_test_emb)

print("Embedding Results")
print(classification_report(y_test, preds_emb))

/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding Results
              precision    recall  f1-score   support

    negative       0.25      1.00      0.40         1
     neutral       1.00      0.50      0.67         2
    positive       0.00      0.00      0.00         2

    accuracy                           0.40         5
   macro avg       0.42      0.50      0.36         5
weighted avg       0.45      0.40      0.35         5



/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/dimtriospanagoulias/miniconda3/envs/nlpunipi/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.

## 6. Compare predictions on custom examples

In [6]:
examples = [
    "I love this product",
    "This is not good",
    "The event starts at 9",
    "Awful support and terrible quality",
]

print("TF-IDF predictions:")
for text, pred in zip(examples, tfidf_clf.predict(examples)):
    print(f"- {text} -> {pred}")

print("\nEmbedding predictions:")
example_emb = embedder.encode(examples, convert_to_numpy=True)
for text, pred in zip(examples, emb_clf.predict(example_emb)):
    print(f"- {text} -> {pred}")

TF-IDF predictions:
- I love this product -> positive
- This is not good -> negative
- The event starts at 9 -> negative
- Awful support and terrible quality -> negative

Embedding predictions:
- I love this product -> positive
- This is not good -> negative
- The event starts at 9 -> neutral
- Awful support and terrible quality -> negative


## 7. Semantic similarity with cosine similarity

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "I love this movie",
    "This film is great",
    "I hate this movie",
    "The weather is sunny today",
]

embeddings = embedder.encode(sentences, convert_to_numpy=True)
sim_matrix = cosine_similarity(embeddings)
pd.DataFrame(sim_matrix, index=sentences, columns=sentences)

,I love this movie,This film is great,I hate this movie,The weather is sunny today
I love this movie,1.000000,0.708314,0.714704,0.040470
This film is great,0.708314,1.000000,0.503964,0.049655
I hate this movie,0.714704,0.503964,1.000000,-0.001181
The weather is sunny today,0.040470,0.049655,-0.001181,1.000000


## 8. Feature engineering examples

In [8]:
import re

def handcrafted_features(text: str):
    return {
        "num_exclamations": text.count("!"),
        "num_uppercase_words": sum(1 for w in text.split() if w.isupper() and len(w) > 1),
        "avg_word_len": sum(len(w) for w in text.split()) / max(len(text.split()), 1),
        "has_negation": int(bool(re.search(r"\b(not|no|never|n't)\b", text.lower()))),
    }

feature_demo = pd.DataFrame([handcrafted_features(t) for t in examples], index=examples)
feature_demo

,num_exclamations,num_uppercase_words,avg_word_len,has_negation
I love this product,0,0,4.00,0
This is not good,0,0,3.25,1
The event starts at 9,0,0,3.40,0
Awful support and terrible quality,0,0,6.00,0


## 9. Student exercises

1. Add more examples to the dataset.
2. Try only unigrams in TF-IDF and compare results.
3. Add handcrafted features such as sentiment lexicon counts.
4. Test ambiguous sentences such as:
   - "The bank approved the loan"
   - "They sat on the river bank"
5. Replace the embedding model with another sentence-transformer model.